# P02 — Aprender representaciones retropropagando errores

## 1. Título y paper

**Paper:** *Learning representations by back-propagating errors*  
**Autoría:** David E. Rumelhart, Geoffrey E. Hinton, Ronald J. Williams  
**Año y venue:** 1986 · Nature, 323, 533–536  
**Nivel:** L2 · **Motor:** `backprop`  
**Ficha completa:** [`P02_backpropagation`](../../papers/foundational/P02_backpropagation/README.md)

**Hito:** Un procedimiento práctico para entrenar capas ocultas: la red descubre representaciones intermedias que nadie diseñó.

- [DOI (Nature)](https://doi.org/10.1038/323533a0)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Sin capas ocultas el perceptrón no resuelve XOR; con capas ocultas no se sabía cómo asignar el error a cada peso interno.
2. Ejecutar una implementación mínima de la propuesta: Aplicar la regla de la cadena hacia atrás por el grafo de cómputo para obtener el gradiente de la pérdida respecto de cada peso.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P01


## 4. Intuición

Si el resultado final está mal, ¿de quién es la culpa? Backpropagation reparte la culpa hacia atrás: cada peso recibe una porción del error proporcional a cuánto influyó en él. No es magia, es la regla de la cadena del cálculo aplicada con orden.


## 5. Concepto mínimo

Para una red `x → h = σ(W₁x + b₁) → o = σ(W₂h + b₂)` y pérdida `L = (o − y)²`:

```text
∂L/∂o_in = 2(o − y)·σ'(o_in)               con σ'(z) = σ(z)(1 − σ(z))
∂L/∂W₂   = ∂L/∂o_in · h
∂L/∂h_in = ∂L/∂o_in · W₂ · σ'(h_in)        ← aquí «viaja» el error hacia atrás
∂L/∂W₁   = ∂L/∂h_in · x
```


## 6. Código explicado

El motor del programa implementa esta derivación a mano, sin autograd, para que se vea cada término.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
resultado = run_paper_lab('backprop', seed=7)
show(resultado['result']['loss_history'])
show(resultado['result']['predictions'])

## 7. Predicción antes de ejecutar

1. ¿Bajará la pérdida de forma monótona o habrá una meseta al principio?
2. Con 2 neuronas ocultas, ¿podrá resolver XOR o hará falta más capacidad?
3. ¿Cuánto esperas que difiera el gradiente analítico del numérico: 1e-2, 1e-5 o 1e-10?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('backprop', seed=semilla)['result']
    print(f"semilla {semilla:>2} · pérdida inicial {r['loss_history'][0]['loss']:.5f} "
          f"· final {r['loss_history'][-1]['loss']:.5f} "
          f"· |analítico − numérico| = {r['grad_check']['abs_diff']}")

## 9. Salida interpretable

La verificación numérica del gradiente es la parte más importante de la salida. `|analítico − numérico| ≈ 1e-8` significa que la derivación es correcta. Si diera `1e-2`, el entrenamiento podría *parecer* funcionar y estar optimizando otra cosa.


## 10. Comentario pedagógico

La meseta inicial es real: con pesos pequeños las sigmoides están en su zona lineal y la señal de gradiente es débil. Ese mismo fenómeno, multiplicado por muchas capas, es el gradiente desvaneciente que P03 tendrá que resolver.


## 11. Error o anti-patrón deliberado

Anti-patrón: confiar en un gradiente escrito a mano sin verificarlo. Aquí se compara contra una derivada numérica *mal calculada* (diferencia hacia adelante con ε enorme).


In [ ]:
eps_malo = 1e-1                      # ε demasiado grande: mide una secante, no una tangente
def f(x):
    return (x - 3) ** 2

aprox = (f(2.0 + eps_malo) - f(2.0)) / eps_malo
print('derivada numérica con ε=1e-1 :', aprox, ' (analítica: -2.0)')

## 12. Corrección

La corrección: diferencia **centrada** y un ε intermedio. Muy grande mide otra cosa; muy pequeño se come la precisión de punto flotante.


In [ ]:
for eps in (1e-1, 1e-3, 1e-5, 1e-9, 1e-12):
    centrada = (f(2.0 + eps) - f(2.0 - eps)) / (2 * eps)
    print(f'ε={eps:<8} → {centrada:+.10f}  error={abs(centrada + 2.0):.2e}')

## 13. Desafío guiado

Comprueba que el error del gradiente sube si aumentas la tasa de aprendizaje hasta desestabilizar el entrenamiento.


In [ ]:
r = run_paper_lab('backprop', seed=3)['result']
print('pérdida final:', r['loss_history'][-1]['loss'])
print('predicciones (XOR espera 0,1,1,0):')
for fila in r['predictions']:
    print(' ', fila['x'], '→', fila['pred'], ' (objetivo', fila['y'], ')')

## 14. Desafío autónomo

Reescribe la red con 1 sola neurona oculta y comprueba que ya no resuelve XOR. Después, sustituye la sigmoide por ReLU y observa qué cambia en la meseta inicial. Documenta ambas curvas de pérdida y explica la diferencia en términos de gradiente.


## 15. Evidencia de aprendizaje

Guarda la curva de pérdida, la comprobación numérica del gradiente y una explicación de por qué una red de 9 parámetros resuelve lo que el perceptrón no podía.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P02_backpropagation/README.md) · evaluación formal: [`assessments/papers/P02_backpropagation.md`](../../assessments/papers/P02_backpropagation.md)


## 16. Cierre

Con backpropagation, las capas ocultas dejan de ser un misterio: se pueden entrenar. El problema siguiente aparece al apilar muchas capas —o muchos pasos de tiempo— y ver que el gradiente se apaga en el camino.


## 17. Conexión con el siguiente hito

- P03
- P04

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
